In [4]:
import fitz #pymupdf

In [5]:
file_path ="../data/raw/ncert_pdfs/iesc107.pdf"
doc=fitz.open(file_path)
print("No of pages = ",len(doc))

No of pages =  24


In [6]:
def is_valid_heading(text):
    if not text:
        return False

    if text.isdigit():
        return False

    if len(text) > 80:
        return False

    if text.startswith(("Fig.", "Table", "Example", "Answer")):
        return False

    if text.startswith(("•", "✓", "y")):
        return False

    return True
def read_pdf(pdf_path):
    doc = fitz.open(pdf_path)

    documents = []

    current_chapter_number = ""
    current_chapter_title = ""
    current_section = ""
    current_subsection = ""

    for page_num in range(len(doc)):
        page = doc[page_num]
        page_text = page.get_text("text")

        blocks = page.get_text("dict")["blocks"]

        # Temporary title accumulator for this page
        chapter_title_parts = []
        section_parts = []
        subsection_parts = []

        for block in blocks:
            for line in block.get("lines", []):
                text = " ".join(
                    span["text"].strip()
                    for span in line["spans"]
                    if span["text"].strip()
                )

                if not text:
                    continue

                # Largest font in the line
                size = max(span["size"] for span in line["spans"])

                # Uncomment for debugging
                #print(f"{size:.1f} : {text}")

                # Chapter number (e.g. "7")
                if size >= 60:
                    current_chapter_number = text

                # Chapter title (can span multiple lines)
                elif size >= 25:
                    chapter_title_parts.append(text)

                # Ignore the word "Chapter"
                elif 20 <= size < 25 and text.lower() == "chapter":
                    continue
               
                # Section heading
                elif 14 <= size < 20:
                    if(is_valid_heading(text)):
                        section_parts.append(text)

                # Subsection / callout heading
                elif 12 <= size < 14:
                    if(is_valid_heading(text)):
                         subsection_parts.append(text)

                if section_parts:
                    current_section = " ".join(section_parts)

                if subsection_parts:
                    current_subsection = " ".join(subsection_parts)
                

        # Combine multi-line chapter title
        if chapter_title_parts:
            current_chapter_title = " ".join(chapter_title_parts).strip()

        documents.append({
            "content": page_text,
            "metadata": {
                "page": page_num + 1,
                "chapter_number": current_chapter_number,
                "chapter_title": current_chapter_title,
                "section": current_section,
                "callout": current_subsection,
                "source": pdf_path
            }
        })

    return documents

In [7]:
from gitsource import chunk_documents
documents = read_pdf(file_path)

doc_chunks = chunk_documents(documents,2000,1000)
print("length of document chunk = ",len(doc_chunks))


length of document chunk =  45


In [8]:
from minsearch import Index
index = Index(text_fields=["content","metadata"])
import json
from minsearch import VectorSearch
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
vector_index = VectorSearch()

for doc in doc_chunks:
    if isinstance(doc.get("metadata"), dict):
        doc["metadata"] = json.dumps(doc["metadata"])

for i, chunk in enumerate(doc_chunks):
    chunk["chunk_id"] = i  
    
index.fit(doc_chunks)


c:\Users\LENOVO\OneDrive\Desktop\revathy\mlzoomcamp\capstoneProj\study-goblin\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1898.38it/s]


In [9]:
chunked_texts = [chunk["content"]+" "+json.dumps(chunk["metadata"]) for chunk in doc_chunks]
from tqdm.auto import tqdm

batch_size = 10
vectors = []

for i in tqdm(range(0, len(chunked_texts), batch_size)):
    batch = chunked_texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

#vector_chunks=embed.encode_batch(chunked_texts)
vector_index.fit(
     vectors,
    doc_chunks
)

100%|██████████| 5/5 [00:05<00:00,  1.10s/it]


In [10]:
from dotenv import load_dotenv
load_dotenv()

import sys
from pathlib import Path
from groq import Groq
import os

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# Go up one directory from the notebook
sys.path.append(str(Path.cwd().parent))
from utils.rag_helper import RAGBase
rag_base = RAGBase(index,client,vector_index,model)
#rag_base.rag(question)


In [8]:
QA_PROMPT = """
You are creating an evaluation dataset for a school textbook RAG system.

From the passage below, generate exactly:

- 2 factual questions
- 1 definition question
- 1 conceptual question
- 1 application question (if possible)

Requirements:
- make sure questions are answerable from the chunk and makes sense in the context of the passage.
- Every answer must come directly from the passage. If you cannot find an answer in the passage, do not make up an answer.Say "Answer not found in the passage" if the answer is not present.
- Do not use outside knowledge.
- Answers should be one or two sentences.
- Return ONLY a JSON array.

Example:

[
  {{
    "question": "...",
    "answer": "..."
  }}
]

PASSAGE:

{content}
"""

In [9]:
import json

def generate_qa_pairs(chunk):

    prompt = QA_PROMPT.format(content=chunk["content"])

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "Return ONLY a valid JSON array. Do not include markdown or explanations."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    text = response.choices[0].message.content.strip()

    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()

    return json.loads(text)

In [17]:
OUTPUT_FILE = "ground_truth.json"

# Load existing results if they exist
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        ground_truth = json.load(f)
else:
    ground_truth = []

for chunk_id, chunk in enumerate(doc_chunks
                                 ):
    try:
        qa_pairs = generate_qa_pairs(chunk)
        for qa in qa_pairs:
            ground_truth.append({
                "chunk_id": chunk_id,
                "question": qa["question"],
                "expected_answer": qa["answer"],
                "context": chunk["content"],
                "metadata": chunk.get("metadata", {})
            })
        print(f"Processed chunk {chunk_id}")

    except Exception as e:
        print(f"Failed chunk {chunk_id}")
        print(e)
ground_truth

Processed chunk 0
Failed chunk 1
'answer'
Processed chunk 2
Processed chunk 3
Processed chunk 4
Processed chunk 5
Processed chunk 6
Processed chunk 7
Processed chunk 8
Processed chunk 9
Processed chunk 10
Processed chunk 11
Processed chunk 12
Processed chunk 13
Processed chunk 14
Processed chunk 15
Processed chunk 16
Processed chunk 17
Failed chunk 18
'answer'
Processed chunk 19
Processed chunk 20
Processed chunk 21
Processed chunk 22
Processed chunk 23
Processed chunk 24
Processed chunk 25
Processed chunk 26
Processed chunk 27
Processed chunk 28
Processed chunk 29
Processed chunk 30
Processed chunk 31
Processed chunk 32
Processed chunk 33
Processed chunk 34
Processed chunk 35
Processed chunk 36
Processed chunk 37
Processed chunk 38
Processed chunk 39
Processed chunk 40
Processed chunk 41
Processed chunk 42
Processed chunk 43
Processed chunk 44


[{'chunk_id': 0,
  'question': 'What is the capacity to do work?',
  'expected_answer': 'Energy, which is the capacity to do work, lies at the heart of all these ideas and of almost every activity in our daily life.',
  'context': 'Work, Energy, and \nSimple Machines\nChapter \n7\nIn earlier Chapters 4 and 6, you have learnt how forces change the \nmotion of objects, and how kinematic equations and Newton’s laws can \nbe used to analyse motion. But when forces change with time or act in \ncomplicated ways, applying these laws directly can become difficult. Is \nthere a simpler and more powerful way to understand such situations? \nIn this chapter, you will explore the ideas of work, energy and power, \nwhich often allow us to analyse motion and interactions more easily. You \nwill also learn about simple machines, which help us perform tasks with \nless effort and more convenience. These form the building blocks of many \neveryday machines. Energy, which is the capacity to do work, lie

In [10]:
with open("ground_truth.json", "w", encoding="utf-8") as f:
    json.dump(ground_truth, f, indent=2, ensure_ascii=False)

print("Saved", len(ground_truth), "examples.")

NameError: name 'ground_truth' is not defined

In [8]:
import json

with open("ground_truth.json", "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

print("Loaded", len(ground_truth), "examples.")

Loaded 208 examples.


In [28]:
import json
import os

OUTPUT_FILE = "evaluation_results.json"
BATCH_SIZE = 3

# Load existing results if they exist
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        results = json.load(f)
else:
    results = []

# Skip questions that are already processed
start = len(results)

for i in range(start, min(start + BATCH_SIZE, len(ground_truth))):

    sample = ground_truth[i]

    print(f"Processing {i+1}/{len(ground_truth)}")

    generated = rag_base.rag(sample["question"])

    results.append({
        "ground_truth_chunk_id": sample["chunk_id"],
        "retrieved_chunks": generated["retrieved_chunk_ids"],
        "question": sample["question"],
        "expected": sample["expected_answer"],
        "generated": generated["answer"]
    })

# Save after every batch
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved {len(results)} results.")

Processing 52/208
rag search result is  [{'start': 1000, 'content': ' possesses due \nto its motion or position. Let us try to quantify mechanical \nenergy by using the concept of work.\n7.4.1 Kinetic energy\nThe energy possessed by an object due to its motion is called \nkinetic energy. All moving objects possess kinetic energy, \nsuch as a moving bicycle or a rolling ball. \nHow much is the energy possessed by an object by virtue of \nits motion? It is common to define an object that does not move \nto have zero kinetic energy. Consider an object that starts from \nrest and acquires a certain velocity under the influence of a \nforce F (Fig. 7.11). Then by the work-energy theorem, the work \ndone by the force will equal to the energy gained by the object, \nwhich is the kinetic energy of the object in this case.\nInitial velocity = 0\nInitial Kinetic energy = 0\nF\nWork done by force\nW = F\u2009×\u2009s\nFinal velocity = v\nFinal Kinetic energy = K\nDisplacement along the force = s\

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kg6rtdvbf6s8dazgq15vskax` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99918, Requested 1103. Please try again in 14m42.144s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [13]:
import pandas as pd
import json

with open("evaluation_results.json", "r", encoding="utf-8") as f:
    eval_results=json.load(f)
df=pd.DataFrame(eval_results)
df

,ground_truth_chunk_id,retrieved_chunks,question,expected,generated,faithfulness
0,0,"[2, 8]",What is the capacity to do work?,"Energy, which is the capacity to do work, lies...",The capacity to do work is referred to as energy.,0
1,0,"[25, 11]",What is the source of energy for a child to walk?,Food provides energy to walk,I'm not able to find the source of energy for ...,0
2,0,"[4, 8]",What is work?,Let us first understand how to define work.,Work is done on an object when a force is appl...,0
3,0,"[0, 11]","How can we use the ideas of work, energy, and ...","In this chapter, you will explore the ideas of...","The ideas of work, energy, and power often all...",0
4,1,"[2, 1]",How much more work is done to lift 3 bags to t...,3 times more work,Three times more work is done to lift 3 bags t...,0
5,1,"[3, 4]",What is the scientific definition of work done...,The work done by a constant force acting on an...,Work done by a force is defined as 1 joule whe...,0
6,1,"[1, 2]",What happens if the same machine is used three...,It would require 3 times more fuel,If the same machine is used three times in suc...,0
7,1,"[2, 1]",Why does applying a larger force over the same...,This shows that applying a larger force over t...,Applying a larger force over the same distance...,0
8,2,"[1, 2]",What is the work done by a force required to l...,The work required to lift the same bag by 1 m ...,"To find the work done, we need the force appli...",0
9,2,"[1, 2]",What is the work done by a force required to l...,You would have carried out 3 times more work a...,"To find the work done, we need the force appli...",0


In [11]:
df["hit"] = df.apply(
    lambda row: row["ground_truth_chunk_id"] in row["retrieved_chunks"],
    axis=1
)

hit_rate = df["hit"].mean()

print("HitRate using Textsearch :", hit_rate)

HitRate using Textsearch : 0.8461538461538461


In [13]:
def reciprocal_rank(row):
    gt = row["ground_truth_chunk_id"]

    if gt in row["retrieved_chunks"]:
        return 1 / (row["retrieved_chunks"].index(gt) + 1)

    return 0

df["rr"] = df.apply(reciprocal_rank, axis=1)

mrr = df["rr"].mean()

print("Mean Reciprocal Rank (MRR) using Textsearch :", mrr)

Mean Reciprocal Rank (MRR) using Textsearch : 0.5548439407814408


In [14]:
import json
import time
import pandas as pd
from groq import RateLimitError

# ==========================================================
# Configuration
# ==========================================================

INPUT_FILE = "evaluation_results.json"          # Original file (will NOT be modified)
OUTPUT_FILE = "evaluation_results_judged.json"  # New file with LLM judge results

BATCH_SIZE = 3
SLEEP_BETWEEN_BATCHES = 10

# ==========================================================
# Load data
# ==========================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    eval_results = json.load(f)

with open("ground_truth.json", "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

# Map chunk_id -> chunk text
chunk_context = {
    item["chunk_id"]: item["context"]
    for item in ground_truth
}

# ==========================================================
# LLM Judge
# ==========================================================

def judge_metrics(question, context, answer):

    prompt = f"""
You are evaluating a Retrieval-Augmented Generation (RAG) system.

Question:
{question}

Retrieved Context:
{context}

Generated Answer:
{answer}

Evaluate the answer using these definitions.

Faithfulness:
- 1 if EVERY factual claim in the answer is supported by the retrieved context.
- Paraphrasing is allowed.
- Synonyms are allowed.
- Different wording is allowed.
- Do NOT require exact sentence matches.

Relevancy:
- 1 if the answer directly answers the user's question.
- 0 otherwise.

Return ONLY valid JSON.

Example:

{{
    "faithfulness": 1,
    "relevancy": 1
}}
"""

    while True:

        try:

            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {
                        "role": "system",
                        "content": "Return ONLY a valid JSON object."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0
            )

            text = response.choices[0].message.content.strip()

            try:
                result = json.loads(text)

                return (
                    int(result.get("faithfulness", 0)),
                    int(result.get("relevancy", 0))
                )

            except Exception:
                print("\nCould not parse judge output:")
                print(text)
                return 0, 0

        except RateLimitError:
            print("Rate limit reached...waiting 5 seconds.")
            time.sleep(5)

# ==========================================================
# Evaluation
# ==========================================================

for start in range(0, len(eval_results), BATCH_SIZE):

    batch = eval_results[start:start + BATCH_SIZE]

    print(f"\nProcessing batch {start // BATCH_SIZE + 1}")

    for row in batch:

        contexts = []

        for chunk_id in row.get("retrieved_chunks", []):

            context = chunk_context.get(chunk_id)

            if context:
                contexts.append(context)

        retrieved_context = "\n\n".join(contexts)

        faithfulness, relevancy = judge_metrics(
            row["question"],
            retrieved_context,
            row["generated"]
        )

        row["faithfulness"] = faithfulness
        row["relevancy"] = relevancy

    # Save progress after every batch WITHOUT touching original file
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(eval_results, f, indent=2, ensure_ascii=False)

    print(f"Batch completed. Progress saved to '{OUTPUT_FILE}'")

    time.sleep(SLEEP_BETWEEN_BATCHES)

# ==========================================================
# Final scores
# ==========================================================

faithfulness_score = sum(r["faithfulness"] for r in eval_results) / len(eval_results)
relevancy_score = sum(r["relevancy"] for r in eval_results) / len(eval_results)

print("\n===================================")
print(f"Faithfulness Score : {faithfulness_score:.3f}")
print(f"Relevancy Score    : {relevancy_score:.3f}")
print("===================================")

# Save final results
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)

print(f"\nFinal results saved to '{OUTPUT_FILE}'")

# Display sample
df = pd.DataFrame(eval_results)
display(df[["question", "generated", "faithfulness", "relevancy"]].head())


Processing batch 1
Batch completed. Progress saved to 'evaluation_results_judged.json'

Processing batch 2
Batch completed. Progress saved to 'evaluation_results_judged.json'

Processing batch 3
Batch completed. Progress saved to 'evaluation_results_judged.json'

Processing batch 4
Batch completed. Progress saved to 'evaluation_results_judged.json'

Processing batch 5
Batch completed. Progress saved to 'evaluation_results_judged.json'

Processing batch 6
Batch completed. Progress saved to 'evaluation_results_judged.json'

Processing batch 7
Batch completed. Progress saved to 'evaluation_results_judged.json'

Processing batch 8
Batch completed. Progress saved to 'evaluation_results_judged.json'

Processing batch 9
Batch completed. Progress saved to 'evaluation_results_judged.json'

Processing batch 10
Batch completed. Progress saved to 'evaluation_results_judged.json'

Processing batch 11
Batch completed. Progress saved to 'evaluation_results_judged.json'

Processing batch 12
Batch com

,question,generated,faithfulness,relevancy
0,What is the capacity to do work?,The capacity to do work is referred to as energy.,1,1
1,What is the source of energy for a child to walk?,I'm not able to find the source of energy for ...,0,0
2,What is work?,Work is done on an object when a force is appl...,1,1
3,"How can we use the ideas of work, energy, and ...","The ideas of work, energy, and power often all...",1,1
4,How much more work is done to lift 3 bags to t...,Three times more work is done to lift 3 bags t...,1,1


HitRate using TextSearch is 0.92, MRR = 0.63 for top 10 results 
HitRate using VectorSearch is 0.85, MRR =0.55  for top 10 results 
HitRate using HybridSearch is 0.94, MRR =0.63  for top 10 results 
So would choose Hybrid search

===================================
Faithfulness Score : 0.961
Relevancy Score    : 0.922
===================================

Final results saved to 'evaluation_results_judged.json'

In [ ]:
chunked_texts = [chunk["content"]+" "+json.dumps(chunk["metadata"]) for chunk in doc_chunks]
from tqdm.auto import tqdm

batch_size = 10
vectors = []

for i in tqdm(range(0, len(chunked_texts), batch_size)):
    batch = chunked_texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

#vector_chunks=embed.encode_batch(chunked_texts)
vector_index.fit(
     vectors,
    doc_chunks
)